In [1]:
import sys; sys.path.insert(0, '..')
from src.ingest.rama import load_wide, to_long, chronological_split
from src.attack.blind import generar_dataset

long = to_long(load_wide('../data/raw/2025O3.xls'), 'O3')
cca = long[long.station == 'CCA'].reset_index(drop=True)
train, test = chronological_split(cca)

ds = generar_dataset(train, bit=5, tasa=0.05)

print(f"Filas: {len(ds)}   atacadas: {ds.atacado.sum()}")
print(f"\nEfecto del ataque (solo filas atacadas):")
print(ds[ds.atacado == 1].efecto.value_counts().to_string())
print(f"\nDaño: {ds[ds.atacado==1]['daño'].sum():.0f} de {ds.atacado.sum()}")

Filas: 7008   atacadas: 334

Efecto del ataque (solo filas atacadas):
efecto
sin_efecto      237
inflado          66
ocultamiento     31

Daño: 97 de 334


In [2]:
at = ds[ds.atacado == 1].copy()
at['delta'] = at.received_value - at.original_value

print("Direccion del flip (sube o baja el valor):")
print((at.delta > 0).value_counts().to_string())
print(f"\nDelta unico: {at.delta.unique()}")

Direccion del flip (sube o baja el valor):
delta
True     232
False    102

Delta unico: [ 32. -32.]


In [3]:
import pandas as pd
from src.attack.blind import generar_dataset, guardar_csv

resumen = []
for bit in range(13):
    ds = generar_dataset(train, bit=bit, tasa=0.05, seed=42)
    guardar_csv(ds, f'../data/processed/cca_o3_bit{bit:02d}_tasa05.csv')

    at = ds[ds.atacado == 1]
    resumen.append({
        'bit': bit,
        'delta_ppb': 2**bit,
        'atacados': len(at),
        'con_daño': int(at['daño'].sum()),
        'tasa_daño_%': round(at['daño'].mean()*100, 1),
        'inflado': int((at.efecto == 'inflado').sum()),
        'ocultamiento': int((at.efecto == 'ocultamiento').sum()),
    })

pd.DataFrame(resumen)

OSError: Cannot save file into a non-existent directory: '../data/processed'

In [ ]:
ds_nuevo = generar_dataset(train, bit=5, tasa=0.05, seed=42)
ds_viejo = pd.read_csv('../data/processed/cca_o3_bit05_tasa05.csv')

print("Mismos mensajes atacados:",
      (ds_nuevo.atacado.values == ds_viejo.atacado.values).all())
print("Mismos valores recibidos:",
      (ds_nuevo.received_value.values == ds_viejo.received_value.values).all())

Mismos mensajes atacados: True
Mismos valores recibidos: False


In [ ]:
dif = ds_nuevo.received_value.values != ds_viejo.received_value.values
print(f"Filas distintas: {dif.sum()} de {len(dif)}")

comp = pd.DataFrame({
    'original': ds_nuevo.original_value.values[dif],
    'viejo': ds_viejo.received_value.values[dif],
    'nuevo': ds_nuevo.received_value.values[dif],
    'atacado': ds_nuevo.atacado.values[dif],
})
print(comp.head(15).to_string())

Filas distintas: 220 de 7008
    original  viejo  nuevo  atacado
0        NaN    NaN    NaN        0
1        NaN    NaN    NaN        0
2        NaN    NaN    NaN        0
3        NaN    NaN    NaN        0
4        NaN    NaN    NaN        0
5        NaN    NaN    NaN        0
6        NaN    NaN    NaN        0
7        NaN    NaN    NaN        0
8        NaN    NaN    NaN        0
9        NaN    NaN    NaN        0
10       NaN    NaN    NaN        0
11       NaN    NaN    NaN        0
12       NaN    NaN    NaN        0
13       NaN    NaN    NaN        0
14       NaN    NaN    NaN        0


In [ ]:
m = ds_nuevo.received_value.notna() & ds_viejo.received_value.notna()

print(f"Filas con valor: {m.sum()}")
print("Idénticos:",
      (ds_nuevo.received_value[m].values == ds_viejo.received_value[m].values).all())
print("NaN en las mismas posiciones:",
      (ds_nuevo.received_value.isna().values == ds_viejo.received_value.isna().values).all())

Filas con valor: 6788
Idénticos: True
NaN en las mismas posiciones: True


## Escenario B.

In [ ]:
ds_mix = generar_dataset(train, bit=[3, 4, 5], tasa=0.05, seed=42)
guardar_csv(ds_mix, '../data/processed/cca_o3_bits345_tasa05.csv')

at = ds_mix[ds_mix.atacado == 1]

print(f"Mensajes atacados: {len(at)}\n")
print("Reparto de bits:")
print(at.bit.value_counts().sort_index().to_string())

print("\nDaño por bit:")
print(at.groupby('bit')['daño'].agg(['size', 'sum', 'mean']).round(3).to_string())

print(f"\nTasa de daño global: {at['daño'].mean()*100:.1f}%")
print("\nEfecto:")
print(at.efecto.value_counts().to_string())

Mensajes atacados: 334

Reparto de bits:
bit
3    100
4    111
5    123

Daño por bit:
     size   sum   mean
bit                   
3     100  17.0  0.170
4     111  12.0  0.108
5     123  33.0  0.268

Tasa de daño global: 18.6%

Efecto:
efecto
sin_efecto      272
inflado          43
ocultamiento     19


In [ ]:
import sys
from pathlib import Path

# VS Code puede ejecutar desde la raiz o desde notebooks/.
# Se encuentra la raiz del proyecto (la carpeta que contiene src/).
RAIZ = Path.cwd().resolve()
if not (RAIZ / "src").is_dir():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.attack.blind import generar_dataset

# Umbrales operativos confirmados por el asesor
FASE_1_PPB = 155
FASE_2_PPB = 200

# Se estudian los bits con desplazamientos físicamente plausibles.
BITS = range(8)  # 1, 2, 4, 8, 16, 32, 64 y 128 ppb

# Todas las mediciones horarias válidas de O3 de RAMA en 2025.
# Cada flip pasará por encode -> encrypt -> flip -> decrypt -> decode.
red_valida = long.loc[
    long["value"].notna(), ["timestamp", "station", "value"]
].copy()

print(f"Lecturas válidas en toda la red: {len(red_valida):,}")
print(f"Estaciones: {red_valida.station.nunique()}")


NameError: name 'long' is not defined